## This is a notebook for our churn prediction model pipeline

### Connect to Google BigQuery Database

### Look at features - EDA

### Feature engineering - Customer segmentation (Kmeans), Time series analysis

### Model selection

### Model evaulation


In [9]:
import gc
import time
import platform
import os
from dataclasses import dataclass
from typing import Callable, Any, List, Dict

import numpy as np
import pandas as pd


### Configure bigquery credentials

#### you need to download gcloud - run 'brew install --cask google-cloud-sdk' and follow instructions to add to path

### then run 'gcloud auth application-default login' in your terminal and a pop up will appear. sign in with your uw email


In [10]:
PROJECT_ID = 'netflix-user-behavior'
# Example query — replace with your dataset/table
SQL = os.environ.get(
    "BQ_SQL",
    """
    SELECT
    m.*,
    w.*
FROM `netflix-user-behavior.kaggle_uncleaned.movies` AS m
JOIN `netflix-user-behavior.kaggle_uncleaned.watch_history` AS w
    ON m.movie_id = w.movie_id
LIMIT 1000;
    """.strip()
)

print("PROJECT_ID:", PROJECT_ID)
print("SQL preview:\n", SQL[:300], "..." if len(SQL) > 300 else "")

PROJECT_ID: netflix-user-behavior
SQL preview:
 SELECT
    m.*,
    w.*
FROM `netflix-user-behavior.kaggle_uncleaned.movies` AS m
JOIN `netflix-user-behavior.kaggle_uncleaned.watch_history` AS w
    ON m.movie_id = w.movie_id
LIMIT 1000; 


In [11]:
from google.cloud import bigquery

def load_bigquery_to_pandas(project_id: str, sql: str) -> pd.DataFrame:
    client = bigquery.Client(project=project_id)
    job = client.query(sql)
    # Use BigQuery Storage API when available for faster download:
    try:
        from google.cloud import bigquery_storage
        bqstorage = bigquery_storage.BigQueryReadClient()
        df = job.result().to_dataframe(bqstorage_client=bqstorage, create_bqstorage_client=False)
    except Exception:
        df = job.result().to_dataframe()
    return df

pdf = load_bigquery_to_pandas(PROJECT_ID, SQL)

ModuleNotFoundError: No module named 'google'